In [ ]:
import pandas as pd
import numpy as np
import time

from scipy.integrate import odeint
import matplotlib.pyplot as plt
import pickle

_ = np.random.seed(0)

import pyabc
from pyabc import ABCSMC, Distribution, RV, MultivariateNormalTransition, QuantileEpsilon
import tempfile

pyabc.settings.set_figure_params('pyabc') 

In [ ]:
def ode_system(y, t, beta, kappa, gamma, N):
    S, E, I, R = y
    dSdt = -beta * S * I / N
    dEdt = beta * S * I / N - kappa * E
    dIdt = kappa * E - gamma * I
    dRdt = gamma * I
    return [dSdt, dEdt, dIdt, dRdt]

def simulate(parameters, ic, T):
    beta, kappa, gamma = parameters
    S0, E0, I0, R0 = ic

    N  = S0 + E0 + I0 + R0
    y0 = ic
    
    t = np.arange(T+1)
    ret = odeint(ode_system, y0, t, args=(beta, kappa, gamma, N))

    return ret[1:]

def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return np.asarray(with_noise)

In [ ]:
N  = 100000
E0 = 0
I0 = 10
R0 = 0
S0 = N - E0 - I0 - R0

init_cond = [S0, E0, I0, R0]

duration=100

In [ ]:
with open('../../Data/Model1/sim_dataset.pkl', 'rb') as f:
    simulation_dataset = pickle.load(f)

In [ ]:
def model(params):
    pvec = [
        params['beta'],
        params['kappa'],
        params['gamma']
    ]
    sim = simulate(pvec, init_cond, T=duration)
    
    return {'cases1': poisson_noise(params['kappa'] * sim[:,1])}

In [ ]:
eps = QuantileEpsilon(initial_epsilon='from_sample', alpha=0.2)
transition = MultivariateNormalTransition(scaling=0.5)

In [ ]:
p_distance = pyabc.AdaptivePNormDistance(
    p=2,
    scale_function=pyabc.distance.std,
    all_particles_for_scale=True
)

In [ ]:
prior = Distribution(
    beta=RV("uniform", 0.01, 1.49),
    kappa=RV("uniform", 0.01, 0.49),
    gamma=RV("uniform", 0.01, 0.49)
)

In [ ]:
param_names = ["beta", "kappa","gamma"]

In [ ]:
results = []

for i, sim in enumerate(simulation_dataset):
    obs_data = sim['poisson']
    obs_dict={"cases1": obs_data}
 
    db_path = "sqlite:///" + tempfile.mkstemp(suffix=f"abc_{i}.db")[1]

    abc = ABCSMC(model, prior, p_distance, transitions=transition, eps=eps, population_size=1000)
    abc.new(db_path, obs_dict)

    history = abc.run(max_total_nr_simulations=100000)

    
    results.append({
        "params": sim['params'],
        "posterior": df_posterior,
        "populations": history.get_all_populations(),
    })

    df_posterior = history.get_distribution()

In [ ]:
posterior_samples=[]

for i, res in enumerate(results):
    df, weights = res['posterior']
    df = df[param_names]
    
    kde_estimator = pyabc.transition.MultivariateNormalTransition()
    kde_estimator.fit(df, weights)
    
    num_samples = 10000
    samples = kde_estimator.rvs(num_samples)
    posterior_samples.append(samples)